# Lecture 3: Linear Operators and Self-Adjoint Matrices
### Computational Companion — Matrix Elements, Eigendecomposition, and the Pauli Matrices

This notebook verifies every claim in Lecture 3 with explicit Python computations:

1. **Resolution of identity** — verify $\sum_i \mathbf{e}_i \mathbf{e}_i^\dagger = I$ and use it to expand vectors
2. **Matrix elements** — compute $M_{ij} = \mathbf{e}_i^\dagger M \mathbf{e}_j$ and verify matrix–vector multiplication
3. **Adjoint and self-adjoint** — verify $(M^\dagger)_{ij} = M_{ji}^*$ and the numbers ↔ matrices analogy
4. **Spectral theorem** — real eigenvalues, orthogonal eigenvectors, completeness
5. **Reverse-engineering Pauli matrices** — reconstruct Z, X, Y from their eigenvectors
6. **Algebraic properties** — $X^2 = I$, $XY = iZ$, commutator relations, $\mathfrak{su}(2)$
7. **Basis invariance** — eigenvalues don't change under basis change; matrix elements do

**Reference:** Lecture 3 notes ("Quantum Systems via Linear Algebra")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

# ── Pauli matrices ──
I2 = np.eye(2, dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)

# ── Basis vectors and eigenvectors ──
e0 = np.array([[1], [0]], dtype=complex)
e1 = np.array([[0], [1]], dtype=complex)

x_plus  = (e0 + e1) / np.sqrt(2)
x_minus = (e0 - e1) / np.sqrt(2)
y_plus  = (e0 + 1j*e1) / np.sqrt(2)
y_minus = (e0 - 1j*e1) / np.sqrt(2)

print("Setup complete: Pauli matrices, basis vectors, and eigenvectors defined.")

## 1. Resolution of Identity

The resolution of identity $\sum_i \mathbf{e}_i \mathbf{e}_i^\dagger = I$ is the fundamental computational tool. Each term $\mathbf{e}_i \mathbf{e}_i^\dagger$ is a rank-1 projector. We verify this for the standard basis and for the X-eigenbasis.

In [ ]:
# ── 1. Resolution of Identity ──

print("=" * 65)
print("RESOLUTION OF IDENTITY: Σ |e_i⟩⟨e_i| = I")
print("=" * 65)

# Standard basis {e0, e1}
proj_0 = e0 @ e0.conj().T
proj_1 = e1 @ e1.conj().T
res_id_std = proj_0 + proj_1

print("\nStandard basis {e0, e1}:")
print(f"  |e0⟩⟨e0| =\n{proj_0.real}")
print(f"  |e1⟩⟨e1| =\n{proj_1.real}")
print(f"  Sum = I ? {np.allclose(res_id_std, I2)}")

# X-eigenbasis {x+, x-}
proj_xp = x_plus @ x_plus.conj().T
proj_xm = x_minus @ x_minus.conj().T
res_id_x = proj_xp + proj_xm

print("\nX-eigenbasis {x+, x-}:")
print(f"  |x+⟩⟨x+| =\n{proj_xp.real}")
print(f"  |x-⟩⟨x-| =\n{proj_xm.real}")
print(f"  Sum = I ? {np.allclose(res_id_x, I2)}")

# Y-eigenbasis {y+, y-}
proj_yp = y_plus @ y_plus.conj().T
proj_ym = y_minus @ y_minus.conj().T
res_id_y = proj_yp + proj_ym
print(f"\nY-eigenbasis: Sum = I ? {np.allclose(res_id_y, I2)}")

# Use resolution of identity to expand a vector
print("\n── Using resolution of identity to expand a vector ──")
v = np.array([[3/5 + 4j/5], [1j]], dtype=complex)
v = v / np.linalg.norm(v)
print(f"\n|v⟩ = {v.T[0]}")

# Expand in standard basis: v = Σ (e_i† v) e_i
alpha_0 = (e0.conj().T @ v)[0, 0]
alpha_1 = (e1.conj().T @ v)[0, 0]
v_reconstructed = alpha_0 * e0 + alpha_1 * e1
print(f"  α₀ = ⟨e0|v⟩ = {alpha_0:.4f}")
print(f"  α₁ = ⟨e1|v⟩ = {alpha_1:.4f}")
print(f"  α₀|e0⟩ + α₁|e1⟩ = {v_reconstructed.T[0]}")
print(f"  Matches original? {np.allclose(v, v_reconstructed)}")

## 2. Matrix Elements and the Adjoint

Matrix elements $M_{ij} = \mathbf{e}_i^\dagger M \mathbf{e}_j$ fully characterize an operator in a given basis. We verify:
- Computing matrix elements reproduces the original matrix
- The adjoint satisfies $(M^\dagger)_{ij} = M_{ji}^*$ (transpose + conjugate)
- The numbers ↔ matrices analogy table from the lecture

In [ ]:
# ── 2. Matrix Elements and the Adjoint ──

print("=" * 65)
print("MATRIX ELEMENTS: M_ij = ⟨e_i| M |e_j⟩")
print("=" * 65)

basis = [e0, e1]
for name, M in [("Z", Z), ("X", X), ("Y", Y)]:
    M_computed = np.zeros((2, 2), dtype=complex)
    for i in range(2):
        for j in range(2):
            M_computed[i, j] = (basis[i].conj().T @ M @ basis[j])[0, 0]
    print(f"\n  {name} from matrix elements:")
    print(f"    Computed =\n{M_computed}")
    print(f"    Original =\n{M}")
    print(f"    Match: {np.allclose(M_computed, M)}")

# ── Adjoint verification ──
print("\n" + "=" * 65)
print("ADJOINT: (M†)_ij = M_ji*")
print("=" * 65)

M_test = np.array([[2, 6+1j], [7, 4-1j]], dtype=complex)
M_dag_manual = M_test.T.conj()
M_dag_numpy = M_test.conj().T

print(f"\nM =\n{M_test}")
print(f"\nM† (transpose + conjugate) =\n{M_dag_manual}")
print(f"Match np method: {np.allclose(M_dag_manual, M_dag_numpy)}")

# Verify entry by entry
for i in range(2):
    for j in range(2):
        lhs = M_dag_manual[i, j]
        rhs = M_test[j, i].conj()
        print(f"  (M†)_{i}{j} = {lhs:.1f}  ==  M_{j}{i}* = {rhs:.1f}  ✓")

# ── Adjoint defining property: u†(Mv) = (M†u)†v ──
print("\n" + "=" * 65)
print("ADJOINT DEFINING PROPERTY: u†(Mv) = (M†u)†v")
print("=" * 65)

u = np.array([[1+2j], [3-1j]], dtype=complex)
v_test = np.array([[2-1j], [1+3j]], dtype=complex)

lhs = (u.conj().T @ M_test @ v_test)[0, 0]
rhs = ((M_dag_manual @ u).conj().T @ v_test)[0, 0]
print(f"\n  u†(Mv)   = {lhs:.4f}")
print(f"  (M†u)†v  = {rhs:.4f}")
print(f"  Equal: {np.isclose(lhs, rhs)}")

# ── Self-adjoint check for Pauli matrices ──
print("\n" + "=" * 65)
print("SELF-ADJOINT CHECK: M = M†")
print("=" * 65)
for name, M in [("Z", Z), ("X", X), ("Y", Y)]:
    print(f"  {name} = {name}† ? {np.allclose(M, M.conj().T)}")

# ── Analogy table verification ──
print("\n" + "=" * 65)
print("NUMBERS ↔ MATRICES ANALOGY")
print("=" * 65)

z = 3 + 4j
print(f"\n  Number z = {z}")
print(f"  z* = {z.conj()}                        ↔  M† = transpose + conjugate")
print(f"  z = z* (real)? {z == z.conj()}          ↔  M = M† (self-adjoint)")
print(f"  |z|² = z*z = {(z.conj()*z).real:.1f}   ↔  ‖v‖² = v†v")
print(f"  |z| = 1 (unit)? {np.isclose(abs(z), 1)}  ↔  U†U = I (unitary)")

# Unit complex → unitary
z_unit = np.exp(1j * np.pi/3)
U_test = expm(-1j * Z * 0.5)
print(f"\n  Unit complex: z = e^(iπ/3) = {z_unit:.4f}, |z| = {abs(z_unit):.6f}")
print(f"  Unitary: U = e^(-iZt/2), U†U = I ? {np.allclose(U_test.conj().T @ U_test, I2)}")

## 3. Spectral Theorem Verification

For self-adjoint matrices ($M = M^\dagger$), the spectral theorem guarantees:
1. All eigenvalues are **real**
2. Eigenvectors with distinct eigenvalues are **orthogonal**
3. Eigenvectors form a **complete** basis (resolution of identity)

We verify all three properties for Z, X, Y and a general random Hermitian matrix.

In [ ]:
# ── 3. Spectral Theorem Verification ──

print("=" * 65)
print("SPECTRAL THEOREM: real eigenvalues, orthogonal eigenvectors, completeness")
print("=" * 65)

for name, M in [("Z", Z), ("X", X), ("Y", Y)]:
    evals, evecs = np.linalg.eigh(M)
    print(f"\n── {name} ──")
    print(f"  Eigenvalues: {evals}  (all real? {np.all(np.isreal(evals))})")

    # Orthogonality
    ip = evecs[:, 0].conj() @ evecs[:, 1]
    print(f"  ⟨e₁|e₂⟩ = {ip:.6f}  (orthogonal? {np.isclose(ip, 0)})")

    # Completeness: Σ |e_i⟩⟨e_i| = I
    res = sum(np.outer(evecs[:, i], evecs[:, i].conj()) for i in range(2))
    print(f"  Σ|eᵢ⟩⟨eᵢ| = I ? {np.allclose(res, I2)}")

    # Spectral decomposition: M = Σ λ_i |e_i⟩⟨e_i|
    M_recon = sum(evals[i] * np.outer(evecs[:, i], evecs[:, i].conj()) for i in range(2))
    print(f"  M = Σλᵢ|eᵢ⟩⟨eᵢ| ? {np.allclose(M_recon, M)}")

# Random Hermitian matrix
print("\n── Random 4×4 Hermitian matrix ──")
rng = np.random.default_rng(42)
A = rng.standard_normal((4, 4)) + 1j * rng.standard_normal((4, 4))
H_rand = (A + A.conj().T) / 2  # Force Hermitian

evals_r, evecs_r = np.linalg.eigh(H_rand)
print(f"  Eigenvalues: {np.round(evals_r, 4)}  (all real? {np.all(np.isreal(evals_r))})")

# Orthogonality: Q†Q = I
gram = evecs_r.conj().T @ evecs_r
print(f"  Eigenvectors orthonormal (Q†Q = I)? {np.allclose(gram, np.eye(4))}")

# Completeness
res_r = sum(np.outer(evecs_r[:, i], evecs_r[:, i].conj()) for i in range(4))
print(f"  Completeness (Σ|eᵢ⟩⟨eᵢ| = I)? {np.allclose(res_r, np.eye(4))}")

# Spectral decomposition
H_recon = sum(evals_r[i] * np.outer(evecs_r[:, i], evecs_r[:, i].conj()) for i in range(4))
print(f"  H = Σλᵢ|eᵢ⟩⟨eᵢ| ? {np.allclose(H_recon, H_rand)}")

## 4. Reverse-Engineering the Pauli Matrices from Eigenvectors

This is the key payoff of Lecture 3: we derived eigenvectors from symmetry constraints in Lecture 2, and now we **reconstruct the matrices** from those eigenvectors via $M = \sum_i \lambda_i |\mathbf{e}_i\rangle\langle\mathbf{e}_i|$.

We also verify the lecture's step-by-step computation: apply the operator to each basis vector first, then take inner products.

In [ ]:
# ── 4. Reverse-Engineering Pauli Matrices from Eigenvectors ──

print("=" * 65)
print("REVERSE-ENGINEERING: M = Σ λ_i |e_i⟩⟨e_i|")
print("=" * 65)

eigdata = {
    "Z": [(+1, e0), (-1, e1)],
    "X": [(+1, x_plus), (-1, x_minus)],
    "Y": [(+1, y_plus), (-1, y_minus)],
}
pauli = {"Z": Z, "X": X, "Y": Y}

for name in ["Z", "X", "Y"]:
    M_recon = sum(lam * (v @ v.conj().T) for lam, v in eigdata[name])
    print(f"\n── {name} ──")
    print(f"  Reconstructed:\n{M_recon}")
    print(f"  Original:\n{pauli[name]}")
    print(f"  Match: {np.allclose(M_recon, pauli[name])}")

# ── Step-by-step: apply operator to basis vectors, then take inner products ──
print("\n" + "=" * 65)
print("STEP-BY-STEP: X computation (Lecture 3, §7.2)")
print("=" * 65)

# X|e0⟩ = (1/√2)(X|x+⟩ + X|x-⟩) = (1/√2)(|x+⟩ - |x-⟩) = |e1⟩
X_e0 = X @ e0
print(f"\n  X|e0⟩ = {X_e0.T[0]}  ==  |e1⟩ = {e1.T[0]}  ✓ {np.allclose(X_e0, e1)}")

# X|e1⟩ = (1/√2)(X|x+⟩ - X|x-⟩) = (1/√2)(|x+⟩ + |x-⟩) = |e0⟩
X_e1 = X @ e1
print(f"  X|e1⟩ = {X_e1.T[0]}  ==  |e0⟩ = {e0.T[0]}  ✓ {np.allclose(X_e1, e0)}")

# Matrix elements
print(f"\n  X₀₀ = ⟨e0|X|e0⟩ = ⟨e0|e1⟩ = {(e0.conj().T @ X_e0)[0,0]:.0f}")
print(f"  X₀₁ = ⟨e0|X|e1⟩ = ⟨e0|e0⟩ = {(e0.conj().T @ X_e1)[0,0]:.0f}")
print(f"  X₁₀ = ⟨e1|X|e0⟩ = ⟨e1|e1⟩ = {(e1.conj().T @ X_e0)[0,0]:.0f}")
print(f"  X₁₁ = ⟨e1|X|e1⟩ = ⟨e1|e0⟩ = {(e1.conj().T @ X_e1)[0,0]:.0f}")

print("\n" + "=" * 65)
print("STEP-BY-STEP: Y computation (Lecture 3, §7.3)")
print("=" * 65)

# Y|e0⟩ = i|e1⟩
Y_e0 = Y @ e0
print(f"\n  Y|e0⟩ = {Y_e0.T[0]}  ==  i|e1⟩ = {(1j*e1).T[0]}  ✓ {np.allclose(Y_e0, 1j*e1)}")

# Y|e1⟩ = -i|e0⟩
Y_e1 = Y @ e1
print(f"  Y|e1⟩ = {Y_e1.T[0]}  ==  -i|e0⟩ = {(-1j*e0).T[0]}  ✓ {np.allclose(Y_e1, -1j*e0)}")

# Matrix elements
print(f"\n  Y₀₀ = ⟨e0|Y|e0⟩ = i·⟨e0|e1⟩ = {(e0.conj().T @ Y_e0)[0,0]}")
print(f"  Y₀₁ = ⟨e0|Y|e1⟩ = -i·⟨e0|e0⟩ = {(e0.conj().T @ Y_e1)[0,0]}")
print(f"  Y₁₀ = ⟨e1|Y|e0⟩ = i·⟨e1|e1⟩ = {(e1.conj().T @ Y_e0)[0,0]}")
print(f"  Y₁₁ = ⟨e1|Y|e1⟩ = -i·⟨e1|e0⟩ = {(e1.conj().T @ Y_e1)[0,0]}")

## 5. Algebraic Properties — $X^2 = I$, $XY = iZ$, Commutators, and $\mathfrak{su}(2)$

The Pauli matrices satisfy:
- **Involutory:** $X^2 = Y^2 = Z^2 = I$
- **Products:** $XY = iZ$, $YZ = iX$, $ZX = iY$ (and reversed products pick up a minus sign)
- **Commutators:** $[X,Y] = 2iZ$, $[Y,Z] = 2iX$, $[Z,X] = 2iY$ — the $\mathfrak{su}(2)$ Lie algebra
- **Anticommutators:** $\{X,Y\} = \{Y,Z\} = \{Z,X\} = 0$ (distinct Paulis anticommute)

In [ ]:
# ── 5. Algebraic Properties ──

print("=" * 65)
print("INVOLUTORY: σ_k² = I")
print("=" * 65)

for name, M in [("X", X), ("Y", Y), ("Z", Z)]:
    sq = M @ M
    print(f"  {name}² = I ? {np.allclose(sq, I2)}")
    if name == "X":
        print(f"    X² =\n{(X @ X).real}")

# ── Products of distinct Pauli matrices ──
print("\n" + "=" * 65)
print("PRODUCTS: XY = iZ, YZ = iX, ZX = iY")
print("=" * 65)

products = [
    ("X", "Y", X, Y, "iZ",  1j*Z),
    ("Y", "X", Y, X, "-iZ", -1j*Z),
    ("Y", "Z", Y, Z, "iX",  1j*X),
    ("Z", "Y", Z, Y, "-iX", -1j*X),
    ("Z", "X", Z, X, "iY",  1j*Y),
    ("X", "Z", X, Z, "-iY", -1j*Y),
]

for n1, n2, M1, M2, expected_name, expected_mat in products:
    result = M1 @ M2
    print(f"  {n1}·{n2} = {expected_name} ? {np.allclose(result, expected_mat)}")

# ── Commutators: [X,Y] = 2iZ, etc. ──
print("\n" + "=" * 65)
print("COMMUTATORS: [σ_i, σ_j] = 2iε_ijk σ_k  (su(2) Lie algebra)")
print("=" * 65)

comm_data = [
    ("X", "Y", X, Y, "Z", Z),
    ("Y", "Z", Y, Z, "X", X),
    ("Z", "X", Z, X, "Y", Y),
]

for n1, n2, M1, M2, n3, M3 in comm_data:
    comm = M1 @ M2 - M2 @ M1
    expected = 2j * M3
    print(f"\n  [{n1}, {n2}] =\n{comm}")
    print(f"  2i·{n3}   =\n{expected}")
    print(f"  Match: {np.allclose(comm, expected)}")

# ── Anticommutators: {σ_i, σ_j} = 2δ_ij I ──
print("\n" + "=" * 65)
print("ANTICOMMUTATORS: {σ_i, σ_j} = 2δ_ij I")
print("=" * 65)

paulis_dict = {"X": X, "Y": Y, "Z": Z}
for n1 in ["X", "Y", "Z"]:
    for n2 in ["X", "Y", "Z"]:
        anti = paulis_dict[n1] @ paulis_dict[n2] + paulis_dict[n2] @ paulis_dict[n1]
        if n1 == n2:
            print(f"  {{{n1},{n2}}} = 2I ? {np.allclose(anti, 2*I2)}")
        else:
            print(f"  {{{n1},{n2}}} = 0  ? {np.allclose(anti, np.zeros((2,2)))}")

## 6. Basis Invariance — Eigenvalues Don't Change, Matrix Elements Do

The lecture emphasizes: matrix elements $M_{ij}$ depend on the choice of basis, but **eigenvalues are intrinsic** to the operator. We verify this by expressing the same operator in two different bases and showing the eigenvalues are identical while the matrix entries change.

Also verified: trace, determinant, and rank are basis-invariant.

In [ ]:
# ── 6. Basis Invariance ──

print("=" * 65)
print("BASIS INVARIANCE: eigenvalues are intrinsic, matrix elements are not")
print("=" * 65)

# Express X in the standard basis {e0, e1} and in the X-eigenbasis {x+, x-}
# Change-of-basis matrix: columns are X-eigenvectors expressed in standard basis
S = np.hstack([x_plus, x_minus])  # columns = x+, x-
S_inv = S.conj().T  # Since S is unitary, S^{-1} = S†

X_standard = X  # X in standard basis
X_xbasis = S_inv @ X @ S  # X in its own eigenbasis

print(f"\nX in standard basis {{e0, e1}}:\n{X_standard.real}")
print(f"\nX in X-eigenbasis {{x+, x-}}:\n{np.round(X_xbasis.real, 10)}")
print("  (diagonal! eigenvalues +1, -1 on the diagonal)")

# Eigenvalues are the same
evals_std = np.sort(np.linalg.eigvalsh(X_standard))
evals_xb  = np.sort(np.linalg.eigvalsh(X_xbasis))
print(f"\nEigenvalues in standard basis: {evals_std}")
print(f"Eigenvalues in X-eigenbasis:  {evals_xb}")
print(f"Same? {np.allclose(evals_std, evals_xb)}")

# But matrix elements differ
print(f"\nX₀₀ in standard basis: {X_standard[0,0].real}")
print(f"X₀₀ in X-eigenbasis:   {X_xbasis[0,0].real}")
print("Matrix elements changed, eigenvalues didn't!")

# ── Basis-invariant quantities: trace, determinant, rank ──
print("\n" + "=" * 65)
print("BASIS-INVARIANT QUANTITIES")
print("=" * 65)

for name, M in [("Z", Z), ("X", X), ("Y", Y)]:
    tr = np.trace(M).real
    det = np.linalg.det(M).real
    rank = np.linalg.matrix_rank(M)
    evals = np.sort(np.linalg.eigvalsh(M))
    print(f"  {name}:  eigenvalues={evals}  trace={tr:.0f}  det={det:.0f}  rank={rank}")

# Verify in rotated basis
print("\n  After basis change (rotate by π/6):")
theta = np.pi / 6
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]], dtype=complex)
for name, M in [("Z", Z), ("X", X), ("Y", Y)]:
    M_rot = R.conj().T @ M @ R
    tr = np.trace(M_rot).real
    det = np.linalg.det(M_rot).real
    evals = np.sort(np.linalg.eigvalsh(M_rot))
    print(f"  {name}:  eigenvalues={np.round(evals, 6)}  trace={tr:.6f}  det={det:.6f}")

## 7. Connection to the Axioms — Born Rule and Expectation Values

With the spectral theorem verified, we can now connect everything back to the axioms:
- **Axiom 2:** Observables are self-adjoint → real eigenvalues (measurement outcomes)
- **Axiom 3:** Born rule $P(\lambda_i) = |\mathbf{e}_i^\dagger \mathbf{v}|^2$ → probabilities sum to 1 via completeness
- **Expectation value:** $\langle M \rangle = \mathbf{v}^\dagger M \mathbf{v} = \sum_i \lambda_i P(\lambda_i)$

In [ ]:
# ── 7. Connection to the Axioms ──

print("=" * 65)
print("BORN RULE AND EXPECTATION VALUES")
print("=" * 65)

# Arbitrary state
psi = np.array([[np.cos(np.pi/5)],
                [np.exp(1j*np.pi/3) * np.sin(np.pi/5)]], dtype=complex)
print(f"\n|ψ⟩ = {psi.T[0]}")
print(f"‖ψ‖ = {np.linalg.norm(psi):.6f}")

for name, M in [("Z", Z), ("X", X), ("Y", Y)]:
    evals, evecs = np.linalg.eigh(M)

    # Born rule: P(λ_i) = |⟨e_i|ψ⟩|²
    probs = [np.abs((evecs[:, i].conj() @ psi.flatten()))**2 for i in range(2)]
    print(f"\n── σ_{name} ──")
    for i, (lam, p) in enumerate(zip(evals, probs)):
        print(f"  P(λ={lam:+.0f}) = |⟨e_{i}|ψ⟩|² = {p:.6f}")
    print(f"  Σ P(λᵢ) = {sum(probs):.6f}  (must be 1)")

    # Expectation value: two ways
    exp_direct = (psi.conj().T @ M @ psi)[0, 0].real
    exp_born = sum(lam * p for lam, p in zip(evals, probs))
    print(f"  ⟨{name}⟩ = v†Mv = {exp_direct:.6f}")
    print(f"  ⟨{name}⟩ = Σλᵢ P(λᵢ) = {exp_born:.6f}")
    print(f"  Match: {np.isclose(exp_direct, exp_born)}")

# ── Visualization: Born-rule probabilities for all 3 observables ──
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, M) in zip(axes, [("Z", Z), ("X", X), ("Y", Y)]):
    evals, evecs = np.linalg.eigh(M)
    probs = [np.abs((evecs[:, i].conj() @ psi.flatten()))**2 for i in range(2)]
    bars = ax.bar([f"λ={v:+.0f}" for v in evals], probs,
                  color=['steelblue', 'salmon'])
    ax.set_ylim(0, 1)
    ax.set_ylabel("P(λ)")
    exp_val = sum(lam * p for lam, p in zip(evals, probs))
    ax.set_title(f"σ_{name}:  ⟨{name}⟩ = {exp_val:.3f}")
    for bar, p in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, p + 0.03, f"{p:.3f}",
                ha='center', fontsize=11)
plt.suptitle("Born Rule Probabilities for |ψ⟩", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Visualizing Projectors and the Spectral Decomposition

Each term $\lambda_i |\mathbf{e}_i\rangle\langle\mathbf{e}_i|$ in the spectral decomposition $M = \sum_i \lambda_i P_i$ is a scaled rank-1 projector. We visualize these projectors as matrices and show how they sum to the full operator.

In [ ]:
# ── 8. Visualizing Projectors and Spectral Decomposition ──

fig, axes = plt.subplots(3, 4, figsize=(16, 10))

for row, (name, M) in enumerate([("Z", Z), ("X", X), ("Y", Y)]):
    evals, evecs = np.linalg.eigh(M)
    projs = [evals[i] * np.outer(evecs[:, i], evecs[:, i].conj()) for i in range(2)]
    M_sum = projs[0] + projs[1]

    matrices = [projs[0], projs[1], M_sum, M]
    titles = [f"λ₁={evals[0]:+.0f} · P₁", f"λ₂={evals[1]:+.0f} · P₂",
              "λ₁P₁ + λ₂P₂", f"{name} (original)"]

    for col, (mat, title) in enumerate(zip(matrices, titles)):
        ax = axes[row, col]
        # Show real part (imaginary part is only nonzero for Y)
        display = mat.real if np.allclose(mat.imag, 0) else mat
        if np.iscomplexobj(display) and not np.allclose(display.imag, 0):
            # For complex matrices, show magnitude
            display_vals = np.abs(mat)
            im = ax.imshow(display_vals, cmap='viridis', vmin=0, vmax=1)
            for i in range(2):
                for j in range(2):
                    val = mat[i, j]
                    txt = f"{val.real:.1f}" if abs(val.imag) < 0.01 else f"{val:.1f}"
                    ax.text(j, i, txt, ha='center', va='center', color='white', fontsize=11)
        else:
            im = ax.imshow(display.real, cmap='RdBu_r', vmin=-1, vmax=1)
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, f"{display.real[i,j]:+.1f}", ha='center', va='center',
                            color='black', fontsize=11)
        ax.set_xticks([0,1]); ax.set_yticks([0,1])
        ax.set_xticklabels(["0","1"]); ax.set_yticklabels(["0","1"])
        ax.set_title(title, fontsize=10)

plt.suptitle("Spectral Decomposition: M = λ₁|e₁⟩⟨e₁| + λ₂|e₂⟩⟨e₂|", fontsize=14)
plt.tight_layout()
plt.show()

# Lecture 3: Linear Operators and Self-Adjoint Matrices
### Computational Companion — Matrix Elements, Eigendecomposition, and the Pauli Matrices

This notebook verifies every claim in Lecture 3 with explicit Python computations:

1. **Resolution of identity** — verify $\sum_i \mathbf{e}_i \mathbf{e}_i^\dagger = I$ and use it to expand vectors
2. **Matrix elements** — compute $M_{ij} = \mathbf{e}_i^\dagger M \mathbf{e}_j$ and verify matrix–vector multiplication
3. **Adjoint and self-adjoint** — verify $(M^\dagger)_{ij} = M_{ji}^*$ and the numbers ↔ matrices analogy
4. **Spectral theorem** — real eigenvalues, orthogonal eigenvectors, completeness
5. **Reverse-engineering Pauli matrices** — reconstruct Z, X, Y from their eigenvectors
6. **Algebraic properties** — $X^2 = I$, $XY = iZ$, commutator relations, $\mathfrak{su}(2)$
7. **Basis invariance** — eigenvalues don't change under basis change; matrix elements do

**Reference:** Lecture 3 notes ("Quantum Systems via Linear Algebra")